# Deep Feature Quality Comparison

Load the test-run representations produced by `gz_foundation_models.py`, score each
encoder with the 2PCF score, intrinsic dimensionality (ID) score, clustering
accuracy, and Davies-Bouldin score (implemented in `backbone/`), then plot each
metric against the others in paper-quality PNGs saved to `../plots`.

In [8]:
import os
import sys
import glob

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import davies_bouldin_score

sys.path.append("..")
import backbone.AstroMLmodified as AstroMLmod
import backbone.AstroMLmod4 as AstroMLmod
import backbone.Test as test
import backbone.VISUAL as viz

REP_DIR = "/idia/projects/camil/Koketso/galaxy_zoo_representations"
PLOTS_DIR = "../plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

## Load representations

Each model has an unlabeled set (`*_reps.h5`, used for the 2PCF/ID scores) and a
labeled classification-validation set (`*_reps_c.h5`, used for clustering
accuracy and Davies-Bouldin).

In [3]:
def open_embeds(file_path):
    #load an embeddings/labels/ids triple saved by gz_foundation_models.py's h5 test runs
    with h5py.File(file_path, "r") as f:
        rep = np.array(f["embeddings"]).astype(np.float32)
        labels = np.array(f["labels"]).astype(str).tolist()
        ids = np.array(f["ids"]).astype(str).tolist()
    return rep, labels, ids


def discover_models(rep_dir):
    #pair each "<name>_reps.h5" with its labeled "<name>_reps_c.h5" counterpart
    models = {}
    for path in sorted(glob.glob(os.path.join(rep_dir, "*_reps.h5"))):
        name = os.path.basename(path)[: -len("_reps.h5")]
        class_path = os.path.join(rep_dir, f"{name}_reps_c.h5")
        if os.path.exists(class_path):
            models[name] = (path, class_path)
    return models


model_paths = discover_models(REP_DIR)
model_paths

{'dinov3_convnext_base': ('/idia/projects/camil/Koketso/galaxy_zoo_representations/dinov3_convnext_base_reps.h5',
  '/idia/projects/camil/Koketso/galaxy_zoo_representations/dinov3_convnext_base_reps_c.h5'),
 'dinov3_vith': ('/idia/projects/camil/Koketso/galaxy_zoo_representations/dinov3_vith_reps.h5',
  '/idia/projects/camil/Koketso/galaxy_zoo_representations/dinov3_vith_reps_c.h5'),
 'imnet_convnext_base': ('/idia/projects/camil/Koketso/galaxy_zoo_representations/imnet_convnext_base_reps.h5',
  '/idia/projects/camil/Koketso/galaxy_zoo_representations/imnet_convnext_base_reps_c.h5'),
 'imnet_resnet18': ('/idia/projects/camil/Koketso/galaxy_zoo_representations/imnet_resnet18_reps.h5',
  '/idia/projects/camil/Koketso/galaxy_zoo_representations/imnet_resnet18_reps_c.h5'),
 'zoobot': ('/idia/projects/camil/Koketso/galaxy_zoo_representations/zoobot_reps.h5',
  '/idia/projects/camil/Koketso/galaxy_zoo_representations/zoobot_reps_c.h5')}

In [4]:
representations = {}
for name, (rep_path, class_path) in model_paths.items():
    rep, label, ids = open_embeds(rep_path)
    rep_c, label_c, ids_c = open_embeds(class_path)
    representations[name] = {
        "rep": rep, "label": label, "ids": ids,
        "rep_c": rep_c, "label_c": label_c, "ids_c": ids_c,
    }
    print(f"{name}: rep={rep.shape}, rep_c={rep_c.shape}")

dinov3_convnext_base: rep=(269760, 1024), rep_c=(7315, 1024)
dinov3_vith: rep=(269760, 1280), rep_c=(7315, 1280)
imnet_convnext_base: rep=(269760, 1024), rep_c=(7315, 1024)
imnet_resnet18: rep=(269760, 512), rep_c=(7315, 512)
zoobot: rep=(269760, 1024), rep_c=(7315, 1024)


## Compute quality metrics

For each model: 2PCF score and ID score on the unlabeled set, clustering
accuracy and Davies-Bouldin score on the labeled classification set (the
latter computed on a 15-component PCA projection, matching `galaxy_zoo_tpcfs.py`).

In [9]:
rows = []
for name, d in representations.items():
    tpcf_mean, tpcf_std = AstroMLmod4.TPCF_score(d["rep"])
    id_mean, id_std = AstroMLmod.id_score(d["rep"])
    accuracy = test.clustering_accuracy((d["rep"], d["ids"]), (d["label_c"], d["ids_c"]))
    davies = davies_bouldin_score(viz.pca(d["rep_c"], n_components=15, verbose=False), d["label_c"])

    print(f"{name}: TPCF={tpcf_mean}, ID={id_mean}, accuracy={accuracy}, davies={davies:.3f}")

    rows.append({
        "model": name,
        "tpcf_mean": tpcf_mean, "tpcf_std": tpcf_std,
        "id_mean": float(np.ravel(id_mean)[0]), "id_std": float(np.ravel(id_std)[0]),
        "clustering_accuracy": accuracy,
        "davies_bouldin": davies,
    })

scores = pd.DataFrame(rows)
scores

AttributeError: module 'backbone.AstroMLmod4' has no attribute 'id_score'

## Plot every metric against every other metric

Paper-quality (300 dpi, serif font) scatter plots, one PNG per metric pair,
saved to `../plots`.

In [ ]:
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

METRICS = {
    "tpcf_mean": ("2PCF Score", "tpcf_std"),
    "id_mean": ("ID Score", "id_std"),
    "clustering_accuracy": ("Clustering Accuracy", None),
    "davies_bouldin": ("Davies-Bouldin Score", None),
}


def paper_scatter(df, x, y, fname):
    #render one metric-pair comparison and save it as a paper-quality png
    xlabel, xerr_col = METRICS[x]
    ylabel, yerr_col = METRICS[y]
    xerr = df[xerr_col] if xerr_col else None
    yerr = df[yerr_col] if yerr_col else None

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.errorbar(df[x], df[y], xerr=xerr, yerr=yerr,
                fmt="o", capsize=4, markersize=8, ecolor="gray", zorder=2)

    for _, row in df.iterrows():
        ax.annotate(row["model"], (row[x], row[y]), textcoords="offset points", xytext=(8, 5))

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{xlabel} vs {ylabel}")
    fig.tight_layout()
    fig.savefig(os.path.join(PLOTS_DIR, fname), bbox_inches="tight")
    plt.show()
    return fig

In [ ]:
from itertools import combinations

for metric_x, metric_y in combinations(METRICS.keys(), 2):
    fname = f"{metric_x}_vs_{metric_y}.png"
    paper_scatter(scores, metric_x, metric_y, fname)
    print(f"Saved: {os.path.join(PLOTS_DIR, fname)}")